# UE -> AIRL -> UE
Clean pipeline: receive → learn → score → return

In [2]:
import socket, json, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from stable_baselines3 import PPO

c:\Users\Danie\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\gym\envs\registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
class UnrealEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.observation_space = gym.spaces.Box(-1, 1, (3,), np.float32)
        self.action_space = gym.spaces.Box(-1, 1, (2,), np.float32)

    def step(self, action):
        return np.zeros(3, np.float32), 0.0, False, False, {}

    def reset(self, seed=None, options=None):
        return np.zeros(3, np.float32), {}

In [4]:
class AIRLDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.reward = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,1))
        self.value = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,1))

    def forward(self, s, s2, logp):
        f = self.reward(s) + 0.99*self.value(s2) - self.value(s)
        return torch.sigmoid(f - logp)

In [ ]:
class UnrealAIRL:

    def __init__(self):
        self.sock = socket.socket()
        self.buffer = []

        self.env = UnrealEnv()
        self.policy = PPO("MlpPolicy", self.env, verbose=0)
        self.disc = AIRLDiscriminator()
        self.opt = optim.Adam(self.disc.parameters(), lr=1e-4)

    def log_pi(self, obs, act):
        o = torch.tensor(obs).float()
        a = torch.tensor(act).float()
        _, lp, _ = self.policy.policy.evaluate_actions(o, a)
        return lp.unsqueeze(-1)

    def rollout(self):
        obs, _ = self.env.reset()
        S,A,S2,LP = [],[],[],[]

        for _ in range(32):
            act,_ = self.policy.predict(obs)
            nxt,_,done,_,_ = self.env.step(act)

            lp = self.log_pi([obs],[act])

            S.append(obs); A.append(act); S2.append(nxt); LP.append(lp.item())
            obs = nxt if not done else self.env.reset()[0]

        return (
            torch.tensor(S).float(),
            torch.tensor(A).float(),
            torch.tensor(S2).float(),
            torch.tensor(LP).unsqueeze(-1)
        )

    def analyze(self):
        if len(self.buffer) < 10:
            return 0.0, [0,0,0]

        data = torch.tensor(self.buffer).float()
        data[:,0] /= 150
        data[:,1] /= 100

        s, s2 = data[:-1], data[1:]
        a = torch.zeros((len(s),2))
        log_exp = self.log_pi(s,a)

        for _ in range(3):
            gs,ga,gs2,log_gen = self.rollout()

            d_exp = self.disc(s,s2,log_exp)
            d_gen = self.disc(gs,gs2,log_gen)

            loss = -(torch.log(d_exp).mean() + torch.log(1-d_gen).mean())

            self.opt.zero_grad(); loss.backward(); self.opt.step()

            self.policy.learn(total_timesteps=64, progress_bar=True)

        print('Model trained!!')

        w = self.disc.reward[0].weight.detach().numpy()
        score = float(0.6*w[0][0] - 0.3*w[0][1] - 0.1*w[0][2])

        return score, w

    def start(self, host="127.0.0.1", port=3000):
        
        print('Session started')
        
        self.sock.bind((host,port))
        self.sock.listen(1)

        while True:
            conn,_ = self.sock.accept()
            raw=""

            with conn:
                
                print('\nConnection found.\n')
                while True:
                    # 1 MB
                    data = conn.recv(1048576)
                    if not data: break
                    

                    raw += data.decode()
                    
                    # print(f'Data found: {raw}')

                    while "\n" in raw:
                        msg,raw = raw.split("\n",1)
                        if not msg.strip(): continue

                        print(f"{msg}") # Print individual packets here

                        pkt = json.loads(msg)

                        if pkt["type"]=="state":
                            self.buffer.append(pkt["data"])

                        elif pkt["type"]=="level_complete":
                            print('Level complete! Analyzing...')
                            score,_ = self.analyze()

                            conn.sendall(json.dumps({
                                "type":"result",
                                "impulsivity_score":score
                            }).encode()+b"\n")
                            
                            print(f'Impulsivity score sent: {score}')

                            self.buffer=[]
                            return

In [6]:
UEAirl = UnrealAIRL()
UEAirl.start()

Session started

Connection found.


Connection found.



KeyboardInterrupt: 